# 02 tokenizer

Step 2 of the guide: every transaction becomes a fixed-order row of tokens, the rows of a card are concatenated in time order and cut into 512-token windows. Each stage below shows what it produced.

```
<TX>  DT_k  HOUR_k  CAT_x  MER_x  AMT_k  DIST_k        7 tokens per transaction
       |      |       |      |      |      |
       |      |       |      |      |      home-to-merchant km, 8 log bins          (switch use_dist)
       |      |       |      |      amount, 20 bins, log or linear, edges fit on train
       |      |       |      merchant if in the train top-500, else MER_OTHER
       |      |       category, one token each
       |      hour of day, 6 periods of 4 h
       minutes since this card's previous transaction, 12 fixed log-spaced bins, DT_NA on a card's first row   (switch use_dt)
```


In [1]:
import os, sys, glob, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
for f in glob.glob("notebooks/*.ipynb"):          # new notebooks from the repo appear in Drive
    dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok")

Mounted at /content/drive
/content/transaction-fm
tfm ok


## What is fit, and on what

Learned from data: amount bin edges, distance bin edges, the category list, the merchant top-N. All of it inside `TransactionTokenizer.fit`, which starts with `assert_train_only` and raises if a val or test row is present (`tests/test_tokenizer.py::test_fit_refuses_non_train_rows`). Fixed by definition, nothing learned: the DT edges (1 min, 10 min, 30 min, 1 h, 3 h, 6 h, 12 h, 1 d, 2 d, 5 d, 30 d) and the 6 hour periods. Val and test rows are then encoded with the train vocabulary; whatever they contain that train never saw shows up as `<UNK>` or `MER_OTHER`, and Stage 2c counts how often.


## Stage 2a: fit on train rows, vocabulary

In [ ]:
import json, numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from tfm.data import train_only
from tfm.tokenizer import TransactionTokenizer, make_windows, DT_EDGES_MIN, HOUR_NAMES

df = pd.read_parquet(f"{ROOT}/data/processed/transactions.parquet")
tr = train_only(df)
print("rows:", len(df), " train rows used for fit:", len(tr))

tok = TransactionTokenizer(use_dt=True, use_dist=True, amt_binning="log", n_amt_bins=20, n_dist_bins=8, top_merchants=500).fit(tr)
print("vocab size:", len(tok.vocab), " tokens per transaction:", tok.tokens_per_tx)
print("merchants in train:", tr["merchant"].nunique(), " kept as own token:", len(tok.merchants))
tok.vocab_summary().to_frame("tokens")


## Stage 2b: the bin edges, in human units

In [ ]:
def edge_table(edges, unit, lo_label="-inf"):
    lo = [lo_label] + [f"{e:,.2f}" for e in edges]
    hi = [f"{e:,.2f}" for e in edges] + ["inf"]
    return pd.DataFrame({"bin": range(len(edges) + 1), f"from ({unit})": lo, f"to ({unit})": hi})

dt = edge_table(np.array(DT_EDGES_MIN), "min", "0")
dt["token"] = [f"DT_{i:02d}" for i in range(len(DT_EDGES_MIN) + 1)]
print("DT: fixed edges (DT_NA for the first row of a card)"); display(dt)

amt = edge_table(tok.amt_edges, "USD", "0"); amt["token"] = [f"AMT_{i:02d}" for i in range(len(amt))]
amt["train rows"] = np.bincount(tok.amt_bin(tr["amt"].to_numpy()), minlength=len(amt))
print("AMT: edges fit on train, evenly spaced in log1p between the train 1st and 99th percentile"); display(amt)

dist = edge_table(tok.dist_edges, "km", "0"); dist["token"] = [f"DIST_{i:02d}" for i in range(len(dist))]
dist["train rows"] = np.bincount(tok.dist_bin(tr["dist_km"].to_numpy()), minlength=len(dist))
print("DIST: same construction"); dist


## Stage 2c: worked example, one card, six consecutive transactions

Left the raw columns, right the tokens they became. This is the same shape as the Wells Fargo worked-example slide. Then the flat token stream those six rows contribute to the card's sequence.


In [ ]:
cid = int(df["card_id"].value_counts().index[len(df["card_id"].unique()) // 2])   # the median-activity card
one = df[df["card_id"] == cid].iloc[100:106].copy()
one["gap_min"] = one["ts"].diff().dt.total_seconds().div(60).round(1)
ids = tok.encode(df[df["card_id"] == cid].iloc[99:106])[1:]          # include the row before so the first DT is real
names = ["TX", "DT", "HOUR", "CAT", "MER", "AMT", "DIST"]
toks = pd.DataFrame([tok.decode(r) for r in ids], columns=names, index=one.index)
display(pd.concat([one[["ts", "gap_min", "category", "merchant", "amt", "dist_km", "is_fraud"]].round(2), toks], axis=1))
print("flat stream:", " ".join(tok.decode(ids)))
print("ids:        ", ids.ravel().tolist())


## Stage 2d: coverage per split

How much of each split falls into the fallback tokens. `MER_OTHER` is by design (merchants outside the train top-500); `<UNK>` means a category never seen in train; `AMT_00` / `AMT_19` and `DIST_00` / `DIST_07` hold everything beyond the train 1st / 99th percentile, so about 1 percent each on train and more on a split whose distribution moved.


In [ ]:
ids_all = tok.encode(df)
v = tok.vocab
rows = []
for s in ("train", "val", "test"):
    m = (df["split"] == s).to_numpy(); x = ids_all[m]
    rows.append({"split": s, "rows": int(m.sum()),
                 "MER_OTHER": (x[:, 4] == v["MER_OTHER"]).mean(),
                 "CAT_<UNK>": (x[:, 3] == v["<UNK>"]).mean(),
                 "AMT first bin": (x[:, 5] == v["AMT_00"]).mean(), "AMT last bin": (x[:, 5] == v["AMT_19"]).mean(),
                 "DIST first bin": (x[:, 6] == v["DIST_00"]).mean(), "DIST last bin": (x[:, 6] == v["DIST_07"]).mean(),
                 "DT_NA": (x[:, 1] == v["DT_NA"]).mean()})
coverage = pd.DataFrame(rows).set_index("split")
coverage.style.format({c: "{:.2%}" for c in coverage.columns if c != "rows"})


## Stage 2e: why log bins for amount (the switch step 6 will flip)

Same 20 bins, edges evenly spaced in raw dollars instead of log dollars. Each bar is the share of train rows in that bin.


In [ ]:
import matplotlib.pyplot as plt
lin = TransactionTokenizer(amt_binning="linear").fit(tr)
occ_log = np.bincount(tok.amt_bin(tr["amt"].to_numpy()), minlength=20) / len(tr)
occ_lin = np.bincount(lin.amt_bin(tr["amt"].to_numpy()), minlength=20) / len(tr)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
ax[0].bar(range(20), occ_log); ax[0].set_title("log bins: share of train rows per AMT token")
ax[1].bar(range(20), occ_lin); ax[1].set_title("linear bins: same rows")
for a in ax: a.set_xlabel("AMT bin")
plt.tight_layout(); plt.savefig(f"{ROOT}/results/02_amt_bins_log_vs_linear.png", dpi=120); plt.show()
print("largest single bin, log:", f"{occ_log.max():.1%}", " linear:", f"{occ_lin.max():.1%}")
print("empty or near-empty bins (<0.5% of rows), log:", int((occ_log < 0.005).sum()), " linear:", int((occ_lin < 0.005).sum()))


## Stage 2f: windows

Each card's token rows of one split are concatenated in time order and cut into 512-token windows with stride 256 (half overlap). Windows never cross a card and never mix splits: train windows hold train rows only, val windows val rows only. A card with fewer than 512 tokens in a split gives one right-padded window.


In [ ]:
card = df["card_id"].to_numpy(); split = df["split"].to_numpy()
windows, metas = {}, {}
rows = []
for s in ("train", "val", "test"):
    w, m = make_windows(ids_all, card, split, s, seq_len=512, stride=256, pad_id=v["<PAD>"])
    windows[s], metas[s] = w, m
    rows.append({"split": s, "windows": len(w), "padded windows": int((m["n_real"] < 512).sum()),
                 "cards": m["card_id"].nunique(), "tokens (real)": int(m["n_real"].sum()),
                 "transactions per full window": 512 // tok.tokens_per_tx})
win = pd.DataFrame(rows).set_index("split"); display(win)
print("train tokens available for pretraining (rows x tokens_per_tx):", int((split == "train").sum()) * tok.tokens_per_tx)
print("first train window, decoded, first 21 tokens:", " ".join(tok.decode(windows["train"][0][:21])))


## Stage 2g: save

`tokenizer.json` (vocab, edges, config: this file travels with every checkpoint), `tokens_<split>.npy` and `windows_<split>_meta.parquet` under `data/processed/`, and `results/02_tokenizer.json` with the numbers above.


In [ ]:
import os
out = f"{ROOT}/data/processed"
tok.save(f"{out}/tokenizer.json")
for s in windows:
    np.save(f"{out}/tokens_{s}.npy", windows[s]); metas[s].to_parquet(f"{out}/windows_{s}_meta.parquet", index=False)
back = TransactionTokenizer.load(f"{out}/tokenizer.json")
print("reload gives identical encoding:", np.array_equal(back.encode(df.head(5000)), ids_all[:5000]))
res = {"vocab_size": len(tok.vocab), "tokens_per_tx": tok.tokens_per_tx, "config": tok.config,
       "vocab_composition": tok.vocab_summary().to_dict(), "coverage": coverage.to_dict(orient="index"),
       "windows": win.to_dict(orient="index"), "amt_edges_usd": tok.amt_edges.tolist(), "dist_edges_km": tok.dist_edges.tolist(),
       "amt_largest_bin_share": {"log": float(occ_log.max()), "linear": float(occ_lin.max())}}
json.dump(res, open(f"{ROOT}/results/02_tokenizer.json", "w"), indent=2)
print(sorted(os.listdir(out)))


## What to read off the numbers

- `vocab size` should land near 570 with top-500 merchants; the guide's 600 to 900 assumed more merchants than Sparkov's 693.
- `MER_OTHER` share on train tells you how much merchant information the top-500 cut throws away; if it is above 30 percent, raise `top_merchants` in step 6.
- Coverage on test versus train is the first drift signal: a test share in `AMT first/last bin` much above 1 percent means amounts moved; step 6's PSI per field makes this formal.
- `train tokens available` is the number step 3 sizes epochs against (the guide expected about 8M).


## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
    cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
    return cells(a) == cells(b)
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
    repo = f"notebooks/{os.path.basename(f)}"
    if not os.path.exists(repo) or same_code(f, repo):
        shutil.copy(f, repo)
    else:
        print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
!git log --oneline | head -1

In [ ]:
drive.flush_and_unmount()